# LeetCode 3757 — Number of Effective Subsequences bằng Python

- Link: https://leetcode.com/problems/number-of-effective-subsequences/
- Độ khó: **Hard**
- Chủ đề: Bitmask, **bao hàm – loại trừ**, **SOS DP** (sum over subsets / zeta transform), biến đổi Möbius, đếm modulo

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countEffective(self, nums: List[int]) -> int:
```

## 1. Mục tiêu học

1. Chuyển "xoá dãy con làm OR giảm" thành "**phần còn lại** thiếu ít nhất một bit của `T = OR(nums)`".
2. Đếm bằng **bao hàm – loại trừ** trên tập bit `M ⊆ T`: `Σ_{M≠∅} (−1)^{|M|+1} · 2^{f(M)}`, với `f(M)` = số phần tử không chứa bit nào của `M`.
3. Tính `f(M)` cho **mọi** `M` bằng **SOS DP** trong `O(b · 2^b)` (`b` = số bit của `T`, `≤ 20`).
4. Viết SOS DP đủ nhanh trong Python (xử lý theo khối bằng slicing).
5. Nhìn lại bài toán qua cặp biến đổi **zeta / Möbius** trên dàn tập con.

## 2. Đề bài & ví dụ

**Sức mạnh** của mảng = OR mọi phần tử (mảng rỗng có OR = 0). Một dãy con là **hiệu quả** nếu xoá nó làm sức mạnh của phần còn lại **giảm thực sự**. Đếm số dãy con hiệu quả, modulo `10^9 + 7`. (Dãy con phân biệt theo **chỉ số**.)

**Ví dụ 1:** `[1,2,3]` → `3`.

**Ví dụ 2:** `[7,4,6]` → `4`.

**Ví dụ 3:** `[8,8]` → `1`.

**Ví dụ 4:** `[2,2,1]` → `5`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `1 ≤ nums[i] ≤ 10^6` (`< 2^20`).

In [1]:
from typing import List
from operator import add
from functools import reduce
import random, time

MOD = 10**9 + 7
examples = [([1, 2, 3], 3), ([7, 4, 6], 4), ([8, 8], 1), ([2, 2, 1], 5)]
for a, exp in examples:
    print(a, "-> kỳ vọng", exp)

[1, 2, 3] -> kỳ vọng 3
[7, 4, 6] -> kỳ vọng 4
[8, 8] -> kỳ vọng 1
[2, 2, 1] -> kỳ vọng 5


## 3. Kiến thức nền

### 3.1. Chuyển sang "phần còn lại"

Chọn dãy con bị xoá `S` ⇔ chọn phần còn lại `R` = phần bù. Vì `OR(R) ⊆ T` (theo bit) nên `OR(R) < T` ⇔ `OR(R) ≠ T` ⇔ `R` **thiếu ít nhất một bit** của `T`. Đáp án = số tập `R` (kể cả rỗng) có `OR(R) ≠ T`.

In [2]:
nums = [2, 2, 1]
T = reduce(lambda x, y: x | y, nums)
n = len(nums)
cnt = sum(1 for m in range(1 << n) if reduce(lambda x, y: x | y, [nums[i] for i in range(n) if m >> i & 1], 0) != T)
print("T =", T, "| số tập R có OR(R) != T:", cnt)

T = 3 | số tập R có OR(R) != T: 5


### 3.2. Bao hàm – loại trừ

Với bit `j` của `T`, gọi `A_j` = các tập `R` **thiếu** bit `j`. Cần `|∪ A_j|`. Giao `∩_{j∈M} A_j` = các `R` chỉ gồm phần tử không chứa bit nào trong `M` ⇒ có `2^{f(M)}` tập. Vậy:

`answer = Σ_{∅ ≠ M ⊆ T} (−1)^{|M|+1} · 2^{f(M)}`, `f(M) = #{i : nums[i] & M = 0}`.

### 3.3. SOS DP (zeta transform trên tập con)

`g[mask]` = số phần tử `x` với `x ⊆ mask`. Bắt đầu `g[x] = cnt[x]`, rồi với từng bit `i`: mọi `mask` có bit `i` cộng thêm `g[mask ^ (1<<i)]`. Sau `b` vòng, `g[mask] = Σ_{x ⊆ mask} cnt[x]`. Khi đó `f(M) = g[T \ M]`.

In [3]:
def sos(cnt, b):
    g = cnt[:]
    for i in range(b):
        for m in range(1 << b):
            if m >> i & 1:
                g[m] += g[m ^ (1 << i)]
    return g

cnt = [0] * 4
for x in [1, 2, 3]: cnt[x] += 1
print("g =", sos(cnt, 2), "(g[mask] = số phần tử là tập con của mask)")

g = [0, 1, 1, 3] (g[mask] = số phần tử là tập con của mask)


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1 `[1,2,3]`, `T = 3` (bit 0, bit 1):

- `M = {bit0}`: phần tử không chứa bit 0 là `{2}` ⇒ `f = 1` ⇒ `+2^1 = 2`.
- `M = {bit1}`: `{1}` ⇒ `+2`.
- `M = {bit0, bit1}`: không có ⇒ `f = 0` ⇒ `−2^0 = −1`.

Tổng `2 + 2 − 1 = 3`. ✔

Số bit của `T` tối đa 20 ⇒ `2^20 ≈ 10^6` tập `M` — chấp nhận được nếu tính mọi `f(M)` cùng lúc bằng SOS DP.

## 5. Cách 1 — Brute Force: duyệt mọi dãy con

In [4]:
def brute(nums):
    n = len(nums); T = 0
    for x in nums: T |= x
    res = 0
    for mask in range(1 << n):                       # mask = tập bị xoá
        rest = 0
        for i in range(n):
            if not mask >> i & 1:
                rest |= nums[i]
        if rest < T:
            res += 1
    return res % MOD

for a, exp in examples:
    print(brute(a), "kỳ vọng", exp)

3 kỳ vọng 3
4 kỳ vọng 4
1 kỳ vọng 1
5 kỳ vọng 5


**Nút thắt:** `O(2^n · n)` — chỉ dùng cho `n ≤ 15`.

## 6. Ý tưởng thuật toán / bước đột phá

1. `T = OR(nums)`, lấy danh sách bit `bits` của `T` (`b = |bits| ≤ 20`). **Nén** mỗi phần tử thành mặt nạ `b` bit (chỉ giữ các bit của `T` — mọi bit của phần tử đều thuộc `T`).
2. `cnt[c]` = số phần tử có mặt nạ nén `c`. SOS DP ⇒ `g[mask]`.
3. `answer = Σ_{M ≠ 0} (−1)^{popcount(M)+1} · 2^{g[full ^ M]}` với `full = 2^b − 1`.

Độ phức tạp `O(n·b + b·2^b)`.

## 7. Các cách tiếp theo

- **Cách 2:** bao hàm – loại trừ, tính `f(M)` bằng cách **quét mảng** cho từng `M` — `O(2^b · n)`.
- **Cách 3:** bao hàm – loại trừ + **SOS DP** — `O(b · 2^b)` (lời giải nộp).
- **Cách 4:** **zeta rồi Möbius**: `h[mask] = 2^{g[mask]}` = số tập có OR ⊆ mask; Möbius cho số tập có OR **đúng bằng** mask; đáp án `2^n − exact[T]`.

### 7.1. Cách 2 — Bao hàm – loại trừ quét trực tiếp

In [5]:
def ie_scan(nums):
    T = 0
    for x in nums: T |= x
    res = 0
    M = T
    while M:                                          # duyệt mọi tập con khác rỗng của T
        f = sum(1 for x in nums if x & M == 0)
        sign = 1 if bin(M).count("1") % 2 == 1 else -1
        res += sign * pow(2, f, MOD)
        M = (M - 1) & T
    return res % MOD

for a, exp in examples:
    print(ie_scan(a), "kỳ vọng", exp)

3 kỳ vọng 3
4 kỳ vọng 4
1 kỳ vọng 1
5 kỳ vọng 5


**Kỹ thuật** `M = (M − 1) & T` duyệt mọi tập con của `T` theo thứ tự giảm dần — rất hay dùng trong bitmask DP.

### 7.2. Cách 3 — SOS DP (lời giải chính)

In [6]:
def compress(nums):
    T = 0
    for x in nums: T |= x
    bits = [i for i in range(T.bit_length()) if T >> i & 1]
    b = len(bits)
    comp = []
    for x in nums:
        c = 0
        for j, bit in enumerate(bits):
            if x >> bit & 1: c |= 1 << j
        comp.append(c)
    return comp, b

def sos_fast(g, b):
    N = 1 << b
    for i in range(b):
        half = 1 << i; step = half << 1
        if half >= 64:                               # khối lớn: cộng cả lát cắt một lần
            for j in range(0, N, step):
                g[j + half:j + step] = map(add, g[j + half:j + step], g[j:j + half])
        else:
            for m in range(N):
                if m & half:
                    g[m] += g[m ^ half]
    return g

def ie_sos(nums):
    comp, b = compress(nums)
    N = 1 << b; full = N - 1
    g = [0] * N
    for c in comp: g[c] += 1
    sos_fast(g, b)
    pow2 = [1] * (len(nums) + 1)
    for i in range(1, len(nums) + 1): pow2[i] = pow2[i - 1] * 2 % MOD
    res = 0
    par = [0] * N
    for M in range(1, N):
        par[M] = par[M >> 1] ^ (M & 1)
        v = pow2[g[full ^ M]]
        res += v if par[M] else -v
    return res % MOD

for a, exp in examples:
    print(ie_sos(a), "kỳ vọng", exp)

3 kỳ vọng 3
4 kỳ vọng 4
1 kỳ vọng 1
5 kỳ vọng 5


### 7.3. Cách 4 — Zeta rồi Möbius

- `h[mask] = 2^{g[mask]}` = số tập con `R` (kể cả rỗng) có `OR(R) ⊆ mask`.
- Möbius (nghịch đảo SOS, dùng phép trừ): `exact[mask]` = số `R` có `OR(R) = mask`.
- Đáp án = `2^n − exact[full]`.

Cách này tính được **phân phối** OR của mọi tập con — hữu ích cho nhiều biến thể.

In [7]:
def zeta_mobius(nums):
    comp, b = compress(nums)
    N = 1 << b; full = N - 1
    g = [0] * N
    for c in comp: g[c] += 1
    sos_fast(g, b)
    h = [pow(2, v, MOD) for v in g]
    for i in range(b):                               # Möbius: trừ thay vì cộng
        half = 1 << i
        for m in range(N):
            if m & half:
                h[m] = (h[m] - h[m ^ half]) % MOD
    return (pow(2, len(nums), MOD) - h[full]) % MOD

for a, exp in examples:
    print(zeta_mobius(a), "kỳ vọng", exp)

3 kỳ vọng 3
4 kỳ vọng 4
1 kỳ vọng 1
5 kỳ vọng 5


## 8. TRACE từng bước

In [8]:
def trace_ie(nums):
    comp, b = compress(nums)
    T = 0
    for x in nums: T |= x
    print(f"nums = {nums}, T = {T} = {bin(T)}, b = {b}, nén = {comp}")
    N = 1 << b; full = N - 1
    g = [0] * N
    for c in comp: g[c] += 1
    print("cnt =", g)
    sos_fast(g, b)
    print("g (SOS) =", g)
    total = 0
    for M in range(1, N):
        f = g[full ^ M]; s = 1 if bin(M).count("1") % 2 else -1
        total += s * 2 ** f
        print(f"  M = {M:0{b}b}: f(M) = g[{full ^ M:0{b}b}] = {f} -> {'+' if s > 0 else '-'}2^{f} = {s * 2 ** f:>3} | lũy kế {total}")
    print("đáp án:", total % MOD)

trace_ie([2, 2, 1])

nums = [2, 2, 1], T = 3 = 0b11, b = 2, nén = [2, 2, 1]
cnt = [0, 1, 2, 0]
g (SOS) = [0, 1, 2, 3]
  M = 01: f(M) = g[10] = 2 -> +2^2 =   4 | lũy kế 4
  M = 10: f(M) = g[01] = 1 -> +2^1 =   2 | lũy kế 6
  M = 11: f(M) = g[00] = 0 -> -2^0 =  -1 | lũy kế 5
đáp án: 5


In [9]:
trace_ie([7, 4, 6])

nums = [7, 4, 6], T = 7 = 0b111, b = 3, nén = [7, 4, 6]
cnt = [0, 0, 0, 0, 1, 0, 1, 1]
g (SOS) = [0, 0, 0, 0, 1, 1, 2, 3]
  M = 001: f(M) = g[110] = 2 -> +2^2 =   4 | lũy kế 4
  M = 010: f(M) = g[101] = 1 -> +2^1 =   2 | lũy kế 6
  M = 011: f(M) = g[100] = 1 -> -2^1 =  -2 | lũy kế 4
  M = 100: f(M) = g[011] = 0 -> +2^0 =   1 | lũy kế 5
  M = 101: f(M) = g[010] = 0 -> -2^0 =  -1 | lũy kế 4
  M = 110: f(M) = g[001] = 0 -> -2^0 =  -1 | lũy kế 3
  M = 111: f(M) = g[000] = 0 -> +2^0 =   1 | lũy kế 4
đáp án: 4


## 9. Code nộp LeetCode

In [10]:
from operator import add

class Solution:
    def countEffective(self, nums: List[int]) -> int:
        MOD = 10**9 + 7
        n = len(nums)
        T = 0
        for x in nums:
            T |= x
        bits = [i for i in range(T.bit_length()) if T >> i & 1]
        b = len(bits)
        N = 1 << b
        full = N - 1
        g = [0] * N
        for x in nums:
            c = 0
            for j, bit in enumerate(bits):
                if x >> bit & 1:
                    c |= 1 << j
            g[c] += 1
        for i in range(b):                               # SOS DP
            half = 1 << i
            step = half << 1
            if half >= 64:
                for j in range(0, N, step):
                    g[j + half:j + step] = map(add, g[j + half:j + step], g[j:j + half])
            else:
                for m in range(N):
                    if m & half:
                        g[m] += g[m ^ half]
        pow2 = [1] * (n + 1)
        for i in range(1, n + 1):
            pow2[i] = pow2[i - 1] * 2 % MOD
        par = [0] * N
        res = 0
        for M in range(1, N):
            par[M] = par[M >> 1] ^ (M & 1)
            if par[M]:
                res += pow2[g[full ^ M]]
            else:
                res -= pow2[g[full ^ M]]
        return res % MOD

sol = Solution()
for a, exp in examples:
    got = sol.countEffective(a)
    print(a, "->", got, "✔" if got == exp else "✘")

[1, 2, 3] -> 3 ✔
[7, 4, 6] -> 4 ✔
[8, 8] -> 1 ✔
[2, 2, 1] -> 5 ✔


## 10. Vì sao thuật toán đúng

1. **Tương đương:** `S` hiệu quả ⇔ `OR(R) < T` với `R` = phần bù. Vì `OR(R)` là tập con (theo bit) của `T`, `OR(R) < T` ⇔ `OR(R) ≠ T` ⇔ tồn tại bit của `T` không có trong `R`.
2. **Bao hàm – loại trừ:** `|∪_{j} A_j| = Σ_{M≠∅} (−1)^{|M|+1} |∩_{j∈M} A_j|`, và `∩_{j∈M} A_j` = mọi tập con của `{i : nums[i] & M = 0}`, có `2^{f(M)}` phần tử.
3. **SOS DP:** sau khi xử lý bit `0..i`, `g[mask]` = số `x` với `x` khác `mask` chỉ ở các bit `≤ i` và `x ⊆ mask` trên các bit đó (quy nạp). Sau mọi bit, `g[mask] = #{x ⊆ mask}`. `nums[i] & M = 0` ⇔ `nums[i] ⊆ full \ M` ⇒ `f(M) = g[full ^ M]`.
4. **Nén bit:** chỉ các bit của `T` xuất hiện trong phần tử, nên ánh xạ bit giữ nguyên quan hệ tập con.
5. Phép tính modulo: `res` có thể âm trước `% MOD` — Python trả kết quả không âm.

## 11. Phân tích Time / Space Complexity

Với `b = popcount(T) ≤ 20`:

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(2^n · n)` | `O(1)` |
| 2. IE quét | `O(2^b · n)` | `O(1)` |
| 3. IE + SOS | `O(n·b + b·2^b)` | `O(2^b)` |
| 4. Zeta + Möbius | `O(n·b + b·2^b)` | `O(2^b)` |

## 12. Bộ test

In [11]:
METHODS = [brute, ie_scan, ie_sos, zeta_mobius, lambda a: Solution().countEffective(a)]
for a, exp in examples:
    assert all(f(a) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [12]:
# 12.2 ca biên
assert Solution().countEffective([5]) == 1                 # chỉ xoá cả mảng
assert Solution().countEffective([3, 3, 3]) == 1
assert Solution().countEffective([1, 2, 4]) == brute([1, 2, 4]) == 7   # bit rời nhau: mọi dãy con khác rỗng
print("Ca biên: OK")

Ca biên: OK


In [13]:
# 12.3 ngẫu nhiên: so với vét cạn
rng = random.Random(3757)
for _ in range(1000):
    n = rng.randint(1, 11)
    a = [rng.randint(1, rng.choice([3, 7, 31, 200])) for _ in range(n)]
    b = brute(a)
    assert all(f(a) == b for f in METHODS[1:]), a
print("1000 test ngẫu nhiên: OK")

1000 test ngẫu nhiên: OK


In [14]:
# 12.4 test lớn: n = 10^5, T có đủ 20 bit
rng = random.Random(2)
a = [rng.randint(1, 10**6) for _ in range(10**5)]
t0 = time.time(); x = Solution().countEffective(a); t1 = time.time()
print("đáp án:", x, "| thời gian:", round(t1 - t0, 2), "s | khớp zeta/Möbius:", x == zeta_mobius(a))

đáp án: 540035774 | thời gian: 0.9 s | khớp zeta/Möbius: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm theo **giá trị** thay vì theo chỉ số

Dãy con phân biệt theo chỉ số: `[2,2,1]` có hai dãy `[2,1]` khác nhau.

In [15]:
def wrong_distinct(nums):
    return brute(sorted(set(nums)))                     # SAI: gộp phần tử trùng

print("sai:", wrong_distinct([2, 2, 1]), "| đúng:", brute([2, 2, 1]))

sai: 3 | đúng: 5


### Lỗi 2 — Duyệt mọi `M` trên 20 bit thay vì chỉ các tập con của `T`

Nếu `M` chứa bit không thuộc `T`, `A_j` cho bit đó là **mọi** tập `R` (không ai có bit đó) ⇒ bao hàm – loại trừ cộng sai.

In [16]:
def wrong_all_bits(nums, B=4):
    res = 0
    for M in range(1, 1 << B):
        f = sum(1 for x in nums if x & M == 0)
        res += (1 if bin(M).count("1") % 2 else -1) * 2 ** f      # SAI: M không giới hạn trong T
    return res % MOD

print("sai:", wrong_all_bits([1, 2, 3]), "| đúng:", brute([1, 2, 3]))

sai: 8 | đúng: 3


### Lỗi 3 — Nhầm dấu bao hàm – loại trừ

In [17]:
def wrong_sign(nums):
    T = 0
    for x in nums: T |= x
    res = 0; M = T
    while M:
        f = sum(1 for x in nums if x & M == 0)
        res += (1 if bin(M).count("1") % 2 == 0 else -1) * 2 ** f   # SAI: đảo dấu
        M = (M - 1) & T
    return res % MOD

print("sai:", wrong_sign([1, 2, 3]), "| đúng:", brute([1, 2, 3]))

sai: 1000000004 | đúng: 3


### Lỗi 4 — SOS DP lặp sai thứ tự (cộng hai lần trong cùng một bit)

Cập nhật kiểu "kéo" từ `mask ^ bit` là đúng; nhưng nếu với mỗi `mask` lại duyệt **mọi** bit rồi cộng ngay (vòng ngoài là mask, vòng trong là bit) thì một tập con bị đếm nhiều lần.

In [18]:
def wrong_sos(cnt, b):
    g = cnt[:]
    for m in range(1 << b):                     # SAI: vòng ngoài là mask
        for i in range(b):
            if m >> i & 1:
                g[m] += g[m ^ (1 << i)]
    return g

cnt = [1, 1, 1, 1]
print("sai:", wrong_sos(cnt, 2), "| đúng:", sos(cnt, 2))

sai: [1, 2, 2, 5] | đúng: [1, 2, 2, 4]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "OR giảm / thiếu bit" ⇒ làm việc với **phần bù** và **bao hàm – loại trừ trên bit**.
- "Số phần tử là tập con của mask, cho mọi mask" ⇒ **SOS DP**.
- `2^{g[mask]}` = số tập có OR ⊆ mask; Möbius ⇒ OR **đúng bằng** mask.
- Python: SOS trên `2^20` cần tối ưu (slicing theo khối) — hoặc nén chỉ các bit thực sự xuất hiện.

## 15. Biến thể của bài

1. **Đếm số tập con có OR = mask cho mọi mask** (Cách 4).
2. **AND thay cho OR**: "xoá làm AND tăng" ⇒ đổi vai trò bit bằng phép bù.
3. **Dãy con liên tiếp** (mảng con) thay vì dãy con: dùng kỹ thuật "OR của mảng con kết thúc tại `i` có ≤ 20 giá trị".
4. **Đếm theo kích thước**: thêm chiều số phần tử còn lại ⇒ đa thức thay cho `2^f`.

In [19]:
# Biến thể 1: phân phối OR của mọi tập con (kể cả rỗng), đối chiếu vét cạn
def or_distribution(nums, B):
    N = 1 << B; g = [0] * N
    for x in nums: g[x] += 1
    sos_fast(g, B)
    h = [2 ** v for v in g]
    for i in range(B):
        for m in range(N):
            if m >> i & 1: h[m] -= h[m ^ (1 << i)]
    return h

rng = random.Random(15)
for _ in range(200):
    a = [rng.randint(0, 7) for _ in range(rng.randint(1, 8))]
    dist = [0] * 8
    for mask in range(1 << len(a)):
        o = 0
        for i in range(len(a)):
            if mask >> i & 1: o |= a[i]
        dist[o] += 1
    assert or_distribution(a, 3) == dist
print("Biến thể 1 OK; ví dụ [1,2,3]:", or_distribution([1, 2, 3], 2))

Biến thể 1 OK; ví dụ [1,2,3]: [1, 1, 1, 5]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Vét cạn | Mọi dãy con | `O(2^n n)` | Kiểm chứng |
| 2. IE quét | Tính `f(M)` trực tiếp | `O(2^b n)` | `b` nhỏ |
| 3. IE + SOS | `f(M) = g[full ^ M]` | `O(b 2^b)` | **Lời giải chuẩn** |
| 4. Zeta + Möbius | OR đúng bằng mask | `O(b 2^b)` | Cần phân phối OR |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `countEffective([4, 4, 4, 4])` = ?

**Bài 2 (tính tay).** `countEffective([1, 1, 2])` bằng bao hàm – loại trừ.

**Bài 3 (điền TODO).** Viết `count_or_equal(nums, target)` = số dãy con (khác rỗng) có OR đúng bằng `target`.

**Bài 4 (tự viết lại).** Viết lại SOS DP dạng "đẩy" (push) thay cho "kéo" (pull) và kiểm tra.

**Bài 5 (mini challenge).** Đếm số dãy con **không rỗng** mà xoá nó làm AND của phần còn lại **tăng** thực sự (AND của mảng rỗng quy ước là `2^20 − 1`).

In [20]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def count_or_equal(nums, target):
    # TODO: chỉ giữ phần tử là tập con của target, rồi bao hàm – loại trừ trên bit của target
    pass

In [21]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def count_and_increase(nums):
    # TODO: lấy phần bù 20 bit rồi quy về bài OR
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `T = 4`, một bit. (2) Chỉ `R = ∅` thiếu bit ⇒ phải xoá cả mảng. (3) `1`.

**Bài 2.** (1) `T = 3`. (2) `M = bit0`: `f = 1` (+2); `M = bit1`: `f = 2` (+4); `M = cả hai`: `f = 0` (−1). (3) `5`.

**Bài 3.** (1) Lọc `x ⊆ target`. (2) `Σ_{M ⊆ target} (−1)^{|M|} 2^{f(M)}` đếm cả tập rỗng. (3) Trừ 1 nếu `target = 0`.

**Bài 4.** (1) Duyệt bit ngoài cùng. (2) Với `m` không có bit `i`: `g[m | bit] += g[m]`. (3) So kết quả.

**Bài 5.** (1) Đặt `y = ~x & (2^20−1)`. (2) AND của `R` tăng ⇔ OR của `y` trên `R` giảm. (3) Gọi lời giải OR (chú ý quy ước mảng rỗng khớp).

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
assert Solution().countEffective([4, 4, 4, 4]) == 1 == brute([4, 4, 4, 4])
assert Solution().countEffective([1, 1, 2]) == 5 == brute([1, 1, 2])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [23]:
# Bài 3, 4
def count_or_equal_sol(nums, target):
    pool = [x for x in nums if x & ~target == 0]
    res = 0; M = target
    while True:
        f = sum(1 for x in pool if x & M == 0)
        res += (-1) ** bin(M).count("1") * 2 ** f
        if M == 0: break
        M = (M - 1) & target
    return res - (1 if target == 0 else 0)          # bỏ tập rỗng

def sos_push(cnt, b):
    g = cnt[:]
    for i in range(b):
        for m in range(1 << b):
            if not m >> i & 1:
                g[m | 1 << i] += g[m]
    return g

rng = random.Random(4)
for _ in range(300):
    a = [rng.randint(0, 7) for _ in range(rng.randint(1, 8))]; t = rng.randint(0, 7)
    exp = 0
    for mask in range(1, 1 << len(a)):
        o = 0
        for i in range(len(a)):
            if mask >> i & 1: o |= a[i]
        exp += (o == t)
    assert count_or_equal_sol(a, t) == exp
    c = [0] * 8
    for x in a: c[x] += 1
    assert sos_push(c, 3) == sos(c, 3)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [24]:
# Bài 5: AND tăng
FULL20 = (1 << 20) - 1
def count_and_increase_sol(nums):
    comp = [~x & FULL20 for x in nums]
    if all(c == 0 for c in comp):                   # AND toàn mảng đã là FULL20: không thể tăng
        return 0
    return Solution().countEffective([c for c in comp]) if all(c > 0 for c in comp) else ie_scan(comp)

def and_brute(nums):
    n = len(nums); A = FULL20
    for x in nums: A &= x
    res = 0
    for mask in range(1, 1 << n):
        r = FULL20
        for i in range(n):
            if not mask >> i & 1: r &= nums[i]
        res += r > A
    return res % MOD

rng = random.Random(5)
for _ in range(300):
    a = [FULL20 ^ rng.randint(0, 7) for _ in range(rng.randint(1, 8))]
    assert count_and_increase_sol(a) == and_brute(a), a
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
T = OR(nums); bits = các bit của T; nén mỗi x về b bit
g[c] += 1 cho mỗi x; SOS: với mỗi bit i, g[m] += g[m ^ (1<<i)] nếu m có bit i
ans = Σ_{M≠0} (-1)^{popcount(M)+1} * 2^{g[full ^ M]}   (mod 1e9+7)
```

**Checklist:**

- [ ] Chuyển sang phần còn lại `R`, kể cả `R = ∅`?
- [ ] `M` chỉ duyệt tập con của `T`?
- [ ] Dấu `(−1)^{|M|+1}`?
- [ ] SOS: vòng ngoài là **bit**, vòng trong là mask?
- [ ] Phần tử trùng giá trị vẫn đếm riêng (theo chỉ số)?